# Tratamento silver — Cota

Transforma o CSV bruto de cota (`data/bronze/ana/cota/cota_todas_estacoes.csv`),
que tem uma linha por **estação/mês** com uma coluna pra cada dia do mês
(`Cota_01`...`Cota_31`), numa série **diária** "longa": uma linha por
estação/dia. Mesma lógica do `tratar_chuva.ipynb`, só com os nomes de
coluna do endpoint de cota (que usam capitalização diferente: `Tipo_Medicao_Cotas`,
`nivelconsistencia` em minúsculo).

Saída: `data/silver/ana/cota/cota_diaria.csv`

Rode as células em ordem, de cima pra baixo.

In [ ]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_do_projeto(marcador: str = "pyproject.toml") -> Path:
    """Sobe a partir da pasta atual até achar o pyproject.toml do projeto.

    Assim funciona não importa em qual pasta este notebook esteja de fato —
    evita o bug clássico de calcular a raiz errado quando o notebook é
    movido de lugar.
    """
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(
        f"Não encontrei '{marcador}' subindo a partir de {atual}. "
        "Este notebook está dentro da pasta do projeto?"
    )


RAIZ = encontrar_raiz_do_projeto()

ENTRADA_CSV = RAIZ / "data" / "bronze" / "ana" / "cota" / "cota_todas_estacoes.csv"
SAIDA_CSV = RAIZ / "data" / "silver" / "ana" / "cota" / "cota_diaria.csv"

print("Raiz do projeto:", RAIZ)
print("Lendo de:", ENTRADA_CSV)
print("Vai salvar em:", SAIDA_CSV)

## 1. Ler o CSV bruto

In [ ]:
bruto = pd.read_csv(ENTRADA_CSV, dtype=str)
print(f"{len(bruto)} linha(s) bruta(s) (uma por estação/mês) lida(s).")
bruto.head()

## 2. Despivotar: uma coluna por dia -> uma linha por dia

Mesma lógica da chuva: pra cada dia de 1 a 31, pega a coluna `Cota_NN`
(+ `Cota_NN_Status`) e transforma numa "fatia" com colunas `data`/`cota_cm`.

In [ ]:
bruto["ano_mes"] = pd.to_datetime(bruto["Data_Hora_Dado"], errors="coerce")

COLUNAS_FINAIS = [
    "codigoestacao", "data", "cota_cm", "status",
    "nivel_consistencia", "tipo_medicao",
]

fatias = []
for dia in range(1, 32):
    coluna_valor = f"Cota_{dia:02d}"
    coluna_status = f"Cota_{dia:02d}_Status"
    if coluna_valor not in bruto.columns:
        continue

    fatia = bruto[[
        "codigoestacao", "ano_mes", coluna_valor, coluna_status,
        "nivelconsistencia", "Tipo_Medicao_Cotas",
    ]].copy()
    fatia.columns = [
        "codigoestacao", "ano_mes", "cota_cm", "status",
        "nivel_consistencia", "tipo_medicao",
    ]

    # data real = mês do registro + esse dia específico da coluna
    fatia["data"] = fatia["ano_mes"] + pd.to_timedelta(dia - 1, unit="D")

    # descarta dias que "vazaram" pro mês seguinte (30/31 de fev., etc.)
    fatia = fatia[fatia["data"].dt.month == fatia["ano_mes"].dt.month]
    # descarta dias sem medição (célula em branco -> NaN)
    fatia = fatia[fatia["cota_cm"].notna()]

    fatias.append(fatia[COLUNAS_FINAIS])

print(f"{len(fatias)} fatia(s) (dias) construída(s).")

## 3. Empilhar tudo e ajustar tipos

In [ ]:
diaria = pd.concat(fatias, ignore_index=True)
diaria["cota_cm"] = pd.to_numeric(diaria["cota_cm"], errors="coerce")
diaria["data"] = pd.to_datetime(diaria["data"]).dt.strftime("%Y-%m-%d")

diaria = diaria.drop_duplicates(subset=["codigoestacao", "data"])
diaria = diaria.sort_values(["codigoestacao", "data"]).reset_index(drop=True)

print(f"{len(diaria)} linha(s) diária(s) geradas.")
diaria.head()

## 4. Conferências rápidas (sanity checks)

In [ ]:
checagem = diaria.copy()
checagem["data_dt"] = pd.to_datetime(checagem["data"])

fev_invalido = checagem[(checagem["data_dt"].dt.month == 2) & (checagem["data_dt"].dt.day >= 29)]
duplicatas = checagem.duplicated(subset=["codigoestacao", "data"]).sum()

print("Dias 29+ de fevereiro encontrados (deveria ser baixo/0 fora de anos bissextos):", len(fev_invalido))
print("Duplicatas estação+data (deveria ser 0):", duplicatas)
print("Intervalo de datas:", checagem["data"].min(), "a", checagem["data"].max())
print("Estações distintas:", checagem["codigoestacao"].nunique())

## 5. Salvar em silver

In [ ]:
SAIDA_CSV.parent.mkdir(parents=True, exist_ok=True)
diaria.to_csv(SAIDA_CSV, index=False, encoding="utf-8")
print(f"Salvo em {SAIDA_CSV}")

## Próximos passos

- Rodar `gerar_parquet_cota.ipynb` (gold) em cima desse CSV.
- Um próximo notebook pode juntar chuva + cota por estação + data.